# NetraV4 — medical VQA on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/melvin/netrav4-medical-vqa-98fc/colab/NetraV4-medical-vqa.ipynb)

[`dipankar500/NetraV4`](https://huggingface.co/dipankar500/NetraV4) is a **~4.4B** medical vision-language model (Apache 2.0): a fine-tune of [`Qwen/Qwen3-VL-4B-Instruct`](https://huggingface.co/Qwen/Qwen3-VL-4B-Instruct) trained with reinforcement learning on medical VQA and exam-style reasoning, then merged back to **BF16**. It reads radiology, pathology, clinical photos, and figures together with text, **reasons in `<think>…</think>`**, and puts the final reply in `<answer>…</answer>`. The Hub checkpoint is three BF16 shards, about **8.9 GB**.

**Why this model.** Most 4B VLMs are general chat. NetraV4 is a small open medical VLM that is required to show its steps before it answers, and the [card](https://huggingface.co/dipankar500/NetraV4) reports it ahead of several larger open models on PMC-VQA, VQA-RAD (closed), and SLAKE (closed). This notebook is the card's Transformers path on a free Colab T4 — not vLLM, not a 4,096-token eval run.

> **Research only — not a medical device, not for diagnosis.**
> NetraV4 is a research model. Its answers can be wrong. Do **not** use this notebook (or the model) for diagnosis, treatment, triage, or any clinical decision. Outputs here are demos on public teaching images, not patient care.

**Licence:** Apache 2.0 (same as the Qwen3-VL-4B base).

**Runtime / VRAM assumption:** *Runtime > Change runtime type > T4 GPU*. A free T4 has about **15 GB** of VRAM, **no native bfloat16 tensor cores**, and **no FlashAttention-2**. The card loads `torch_dtype=torch.bfloat16` and generates up to **4,096** new tokens. That is the wrong type and too much KV for 16 GB. This notebook loads **float16 + SDPA** first (~9 GB of weights), falls back to **4-bit NF4** via bitsandbytes if float16 overflows, downscales images so the long side is **768 px** (the processor already caps `max_pixels=802816`), and keeps **`max_new_tokens=512`**. Expected peak: **~11–14 GB** in float16 or **~6–10 GB** in 4-bit — an estimate from the weight files plus one 768 px image, not a measured Colab number yet. Attention is **SDPA** (eager if SDPA refuses the stack). Do not install `flash-attn`.

**Not in this notebook:** vLLM, the card's 4,096-token greedy eval, BF16, FlashAttention-2, or any real clinical workflow.

**Links:** [model card](https://huggingface.co/dipankar500/NetraV4) · [base Qwen3-VL-4B-Instruct](https://huggingface.co/Qwen/Qwen3-VL-4B-Instruct) · [announcement](https://x.com/VaidenceAI/status/2117133765973840126)


## 1. Install Transformers and bitsandbytes

The [model card](https://huggingface.co/dipankar500/NetraV4) uses stock `AutoModelForImageTextToText` + `AutoProcessor` (`Qwen3VLForConditionalGeneration` / `Qwen3VLProcessor`). The checkpoint was saved with `transformers_version: 4.57.1`. Qwen3-VL landed in 4.57; Colab's current 5.x stack also loads it. `Qwen3VLProcessor` imports torchvision even for still images. Use **`%pip`** (not `python -I -m pip`). Isolated mode drops Colab's site-packages and breaks the preinstalled `torch`. Keep Colab's torch. Do **not** install `flash-attn`. `bitsandbytes` is only the 4-bit NF4 fallback if float16 overflows.


In [ ]:
%pip install -q -U "transformers>=4.57.1" "accelerate>=1.1.0" "bitsandbytes>=0.45.0" torchvision pillow

# Card / checkpoint: transformers 4.57.1; Qwen3-VL is in 4.57+ and current 5.x.
# bitsandbytes is the 4-bit NF4 fallback if float16 + a 768 px image overflows ~15 GB.
# torchvision is required by Qwen3VLProcessor (video preprocessor) even for still images.
# Use %pip, not python -I -m pip — isolated mode drops Colab site-packages.
# Do NOT replace Colab's torch. Do NOT install flash-attn.


> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.
> Colab has already imported its preinstalled transformers. The upgrade is invisible until the process restarts.


## 2. Check the GPU and print T4 caveats

The next cell must see a CUDA GPU. It prints `torch.cuda.is_available()`, the device name, and free VRAM, then stops if you are on CPU.


In [ ]:
import gc

import torch
import transformers
from packaging.version import Version

assert Version(transformers.__version__.split("+")[0]) >= Version("4.57.1"), (
    f"transformers {transformers.__version__} is loaded; NetraV4 needs 4.57.1+ "
    "(Qwen3VLForConditionalGeneration). Run the install cell, then Runtime > Restart session."
)

print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

# T4 (sm_75) has no FlashAttention-2 and no native bfloat16 tensor cores.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

DEVICE = "cuda"
major, minor = torch.cuda.get_device_capability()
try:
    NATIVE_BF16 = torch.cuda.is_bf16_supported(including_emulation=False)
except TypeError:
    NATIVE_BF16 = major >= 8
free, total = torch.cuda.mem_get_info()
DTYPE = torch.float16  # card default is bfloat16; T4 has no native bf16

print("transformers", transformers.__version__, "| torch", torch.__version__)
print("device:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
print(f"VRAM free: {free / 1024**3:.1f} GB of {total / 1024**3:.1f} GB")
print("native bf16:", NATIVE_BF16, "| FlashAttention-2: off (sdpa / eager only)")
print("load path: float16 + SDPA first (~9 GB), 4-bit NF4 fallback if VRAM overflows")
print("image cap: long side 768 px | max_new_tokens: 512 (card eval is 4096)")


def print_vram(label=""):
    suffix = f"  ({label})" if label else ""
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    free_now, total_now = torch.cuda.mem_get_info()
    print(
        f"VRAM allocated: {allocated:.2f} GB | reserved: {reserved:.2f} GB | "
        f"peak: {peak:.2f} GB | free: {free_now / 1024**3:.1f}/{total_now / 1024**3:.1f} GB{suffix}"
    )


## 3. Load NetraV4

`AutoModelForImageTextToText` + `AutoProcessor`, matching the [card's usage snippet](https://huggingface.co/dipankar500/NetraV4#usage). The architecture is stock `Qwen3VLForConditionalGeneration`; no `trust_remote_code`. The first run downloads about **8.9 GB**.

The card evaluates with **greedy** decoding (`do_sample=False`) and up to 4,096 new tokens. Demos here stay greedy and **512** new tokens so a T4 finishes and peak VRAM stays under ~14 GB.


In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig

MODEL_ID = "dipankar500/NetraV4"
MAX_SIDE = 768
MAX_NEW_TOKENS = 512
QUANT = None
ATTN = None
model = None
last_error = None

# Card instruction — required. The model was evaluated with this exact format.
INSTRUCTION = (
    "Think step-by-step and enclose your reasoning in <think>...</think> tags. "
    "Then provide your answer in <answer>...</answer> tags."
)

processor = AutoProcessor.from_pretrained(MODEL_ID)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()


def load_vlm(attn, quant=None):
    kwargs = {
        "device_map": "auto",
        "low_cpu_mem_usage": True,
        "attn_implementation": attn,
    }
    if quant is None:
        kwargs["dtype"] = DTYPE
    else:
        kwargs["quantization_config"] = quant
    return AutoModelForImageTextToText.from_pretrained(MODEL_ID, **kwargs).eval()


for attn in ("sdpa", "eager"):
    try:
        print(f"loading float16 + {attn}…")
        model = load_vlm(attn)
        ATTN = attn
        QUANT = None
        break
    except Exception as exc:
        last_error = exc
        print(f"{attn} float16 load failed:", exc)
        try:
            del model
        except NameError:
            pass
        model = None
        gc.collect()
        torch.cuda.empty_cache()

if model is None:
    print("retrying 4-bit NF4 bitsandbytes…")
    QUANT = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    for attn in ("sdpa", "eager"):
        try:
            model = load_vlm(attn, quant=QUANT)
            ATTN = attn
            break
        except Exception as exc:
            last_error = exc
            print(f"{attn} 4-bit load failed:", exc)
            try:
                del model
            except NameError:
                pass
            model = None
            gc.collect()
            torch.cuda.empty_cache()

if model is None:
    raise RuntimeError("could not load dipankar500/NetraV4") from last_error

DEVICE = next(model.parameters()).device
n_params = sum(p.numel() for p in model.parameters()) / 1e9
print("class:", type(model).__name__)
print("processor:", type(processor).__name__)
print("attention:", ATTN, "| dtype:", getattr(model, "dtype", "quantized"))
print("quantization:", "4-bit nf4" if QUANT else "none (float16)")
print(f"parameters: {n_params:.2f}B")
print_vram("after load")


## 4. Helpers — card prompt format and a polite image fetch

The card defines its own contract (do not skip it):

1. User text is `question + "\n" +` the `<think>` / `<answer>` instruction.
2. After `apply_chat_template(..., add_generation_prompt=True)`, **append** `"<think>\n"` so generation starts inside the reasoning block.
3. For multiple-choice, list options after the question, one per line (`A. …`, `B. …`). The model puts the chosen option inside `<answer>…</answer>`.

Wikimedia Commons 403s a default Python User-Agent. The fetch helper sends a descriptive UA and retries transient errors. Images are downscaled so the long side is 768 px before they hit the processor.


In [ ]:
import io
import re
import time

import requests
from IPython.display import display
from PIL import Image

UA = {
    "User-Agent": (
        "donvito-notebooks/1.0 (educational Colab demo; "
        "https://github.com/donvito/notebooks)"
    )
}


def fetch_image(url, max_side=MAX_SIDE):
    for attempt in range(4):
        response = requests.get(url, headers=UA, timeout=120)
        if response.status_code not in (429, 500, 502, 503, 504) or attempt == 3:
            response.raise_for_status()
            image = Image.open(io.BytesIO(response.content)).convert("RGB")
            break
        retry_after = response.headers.get("Retry-After", "")
        time.sleep(int(retry_after) if retry_after.isdigit() else 2 ** (attempt + 1))
    else:
        raise RuntimeError(f"could not download {url}")

    longest = max(image.size)
    if longest > max_side:
        scale = max_side / longest
        image = image.resize(
            (max(1, int(image.width * scale)), max(1, int(image.height * scale))),
            Image.Resampling.LANCZOS,
        )
    return image


def parse_think_answer(text):
    # Split the card's <think> / <answer> tags. Generation starts after '<think>\n'.
    full = text if text.lstrip().startswith("<think>") else "<think>\n" + text
    think = ""
    answer = ""
    m_think = re.search(r"<think>(.*?)</think>", full, flags=re.DOTALL | re.IGNORECASE)
    m_answer = re.search(r"<answer>(.*?)</answer>", full, flags=re.DOTALL | re.IGNORECASE)
    if m_think:
        think = m_think.group(1).strip()
    if m_answer:
        answer = m_answer.group(1).strip()
    return full, think, answer


def ask(image, question, max_new_tokens=MAX_NEW_TOKENS):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": question + "\n" + INSTRUCTION},
        ],
    }]
    prompt = (
        processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        + "<think>\n"
    )
    inputs = processor(text=[prompt], images=[image], return_tensors="pt")
    inputs = inputs.to(DEVICE)
    for key, value in list(inputs.items()):
        if hasattr(value, "is_floating_point") and value.is_floating_point():
            inputs[key] = value.to(dtype=torch.float16 if QUANT else DTYPE)

    torch.cuda.reset_peak_memory_stats()
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    continuation = processor.decode(
        output[0, inputs["input_ids"].shape[1] :],
        skip_special_tokens=True,
    )
    full, think, answer = parse_think_answer(continuation)
    print("Question:")
    print(question)
    print()
    print("Reasoning (<think>):")
    print(think or "(no </think> tag — output may have been cut by max_new_tokens=512)")
    print()
    print("Answer (<answer>):")
    print(answer or "(no <answer> tag — see raw continuation below)")
    if not answer:
        print()
        print("Raw continuation:")
        print(continuation)
    print_vram("after generate")
    return full


## 5. Demo 1 — chest X-ray (card question)

Public-domain **normal PA chest radiograph** ([CC0](https://creativecommons.org/publicdomain/zero/1.0/), Mikael Häggström, M.D., [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Normal_posteroanterior_(PA)_chest_radiograph_(X-ray).jpg)). The question is the card's own example: *Is there evidence of pleural effusion?*

This is a teaching image of a **normal** study, not a patient encounter.


In [ ]:
CXR_URL = (
    "https://upload.wikimedia.org/wikipedia/commons/a/a1/"
    "Normal_posteroanterior_%28PA%29_chest_radiograph_%28X-ray%29.jpg"
)
cxr = fetch_image(CXR_URL)
print("chest X-ray", cxr.size, "| source: Wikimedia Commons, CC0")
display(cxr)

ask(cxr, "Is there evidence of pleural effusion?")


## 6. Demo 2 — histopathology

Public-domain H&E lung section of **severe emphysema** ([CC0](https://creativecommons.org/publicdomain/zero/1.0/), Mikael Häggström, M.D., [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Histopathology_of_emphysema.jpg)). The model should notice the enlarged air spaces before it answers.

Again: research demo only, not a pathology report.


In [ ]:
HISTO_URL = (
    "https://upload.wikimedia.org/wikipedia/commons/1/1d/Histopathology_of_emphysema.jpg"
)
histo = fetch_image(HISTO_URL)
print("histopathology", histo.size, "| source: Wikimedia Commons, CC0")
display(histo)

ask(
    histo,
    "What is the main histopathological finding in this H&E lung section, and what stain is this?",
)


## 7. Demo 3 — multiple-choice exam-style question

CDC public-domain AP chest radiograph of far-advanced pulmonary tuberculosis ([PD-USGov-HHS-CDC](https://commons.wikimedia.org/wiki/File:Tuberculosis-x-ray.jpg), PHIL #2543). Options follow the card: one per line, `A. …` / `B. …`. The model should put the chosen letter (and wording) inside `<answer>…</answer>`.

This is a **historical teaching radiograph**, not a live case.


In [ ]:
TB_URL = "https://upload.wikimedia.org/wikipedia/commons/4/4f/Tuberculosis-x-ray.jpg"
tb = fetch_image(TB_URL)
print("TB teaching radiograph", tb.size, "| source: CDC / Wikimedia Commons, public domain")
display(tb)

mcq = (
    "Which finding is most consistent with this chest radiograph?\n"
    "A. Normal lungs with no focal opacity\n"
    "B. Bilateral pulmonary infiltrates with right apical cavitation\n"
    "C. Isolated left lower-lobe collapse without other findings\n"
    "D. Tension pneumothorax with mediastinal shift to the right"
)
ask(tb, mcq)


## 8. Caveats and cleanup

**Research only — not for diagnosis.** The card is explicit: NetraV4 is not a medical device. Answers can be confident and wrong. Do not use this notebook for diagnosis, treatment, or any clinical decision.

**Prompt contract (from the card).** Every turn is the question, then the `<think>` / `<answer>` instruction, then a primed `"<think>\n"` after the chat template. Multiple-choice options are listed `A.` / `B.` / `C.` / `D.`, one per line. Skipping that format is a different model than the one on the card.

**T4 / VRAM.** Float16 + SDPA first (~9 GB). 4-bit NF4 if load or generate overflows. Images are capped at 768 px on the long side; the processor's `max_pixels` is already 802,816. `max_new_tokens=512` (card eval is 4,096). Do not load bf16. Do not install FlashAttention-2. The last number below is `torch.cuda.max_memory_allocated` for this session.

**Reasoning length.** The card notes some answers need more than 4,096 tokens and get cut off. 512 new tokens is enough to *see* the think/answer tags on a T4; raise it on a larger GPU if a reply stops mid-thought.

**Images.** All three files are license-clean Wikimedia Commons teaching images (CC0 or US government public domain). The fetch helper sends a User-Agent; a bare `requests.get` often 403s.

**Not covered here.** vLLM, the card's 4,096-token greedy eval, video, or any real clinical workflow.

**Licence.** Apache 2.0.

**Links again:** [model card](https://huggingface.co/dipankar500/NetraV4) · [Qwen3-VL-4B-Instruct](https://huggingface.co/Qwen/Qwen3-VL-4B-Instruct) · [announcement](https://x.com/VaidenceAI/status/2117133765973840126)

Free the GPU before you open another notebook:


In [ ]:
print_vram("before cleanup")
try:
    del model
except NameError:
    pass
try:
    del processor
except NameError:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print_vram("after cleanup")
    print(f"peak VRAM this session: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
print("NetraV4 released. Re-run the load cell if you want another prompt.")
